In [ ]:
!pip install torch transformers accelerate bitsandbytes langchain-community langchain-text-splitters langchain-core faiss-cpu pymupdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!pip install rouge-score bert-score nltk pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ca3bc9dda992498438789f8113d9441eba957c024145766a3e415c96d5a58ab4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import torch
import gc
import os
import csv
import time
import re
import string
import collections
import pandas as pd
from datetime import datetime
from typing import List

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModel,
)
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sentence_transformers import CrossEncoder, SentenceTransformer, util

# --- NEW METRIC LIBRARIES ---
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
HF_TOKEN = os.getenv("HF_TOKEN")
PDF_PATH = "/content/2024ESC-compressed.pdf"
INPUT_CSV = "RAG Test Prompts for Medical Guidelines - RAG Test Prompts for Medical Guidelines.csv"
OUTPUT_CSV = "rag_evaluation_qwen_full_metrics.csv"

# ── MedCPT Embeddings ─────────────────────────────────────────────────────────
class MedCPTEmbeddings(Embeddings):
    def __init__(self, load_article_encoder: bool = True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("  Loading MedCPT Query Encoder...")
        self.models = {
            "qry_tok": AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder"),
            "qry_mod": AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder"),
        }
        if load_article_encoder:
            print("  Loading MedCPT Article Encoder...")
            self.models["art_tok"] = AutoTokenizer.from_pretrained(
                "ncbi/MedCPT-Article-Encoder"
            )
            self.models["art_mod"] = AutoModel.from_pretrained(
                "ncbi/MedCPT-Article-Encoder"
            ).to(self.device)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 8
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = self.models["art_tok"](
                batch,
                max_length=512,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)
            with torch.no_grad():
                output = self.models["art_mod"](**inputs)
                all_embeddings.extend(output.last_hidden_state[:, 0, :].cpu().tolist())
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        self.models["qry_mod"].to(self.device)
        inputs = self.models["qry_tok"](
            [text],
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            output = self.models["qry_mod"](**inputs)
            result = output.last_hidden_state[:, 0, :][0].cpu().tolist()
        self.models["qry_mod"].to("cpu")
        del inputs, output
        torch.cuda.empty_cache()
        return result

    def unload_article_encoder(self):
        if "art_mod" in self.models:
            del self.models["art_mod"], self.models["art_tok"]
            torch.cuda.empty_cache()
            gc.collect()
            print("  Article encoder unloaded.")

# ── Build index ───────────────────────────────────────────────────────────────
print("\n⚙️  Loading and indexing PDF with MedCPT...")
loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks = splitter.split_documents(documents)

medcpt_embeddings = MedCPTEmbeddings(load_article_encoder=True)
vector_store = FAISS.from_documents(chunks, medcpt_embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 8})
print(f"  Indexed {len(chunks)} chunks.")

# Free article encoder VRAM before loading LLM
medcpt_embeddings.unload_article_encoder()

# ── Load Reranker & Metrics Models ────────────────────────────────────────────
print("\n⚙️  Loading Reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cpu")

print("⚙️  Loading Semantic Similarity Model...")
metric_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# ── Load LLM ──────────────────────────────────────────────────────────────────
print("\n⚙️  Loading LLM...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, token=HF_TOKEN, trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
)
print("  LLM ready.\n")

# ── Metrics Evaluation Functions ──────────────────────────────────────────────
def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def calc_accuracy_metrics(pred, truth):
    norm_pred = normalize_text(pred).split()
    norm_truth = normalize_text(truth).split()

    exact_match = int(norm_pred == norm_truth)

    common = collections.Counter(norm_pred) & collections.Counter(norm_truth)
    num_same = sum(common.values())

    if len(norm_pred) == 0 or len(norm_truth) == 0:
        f1 = precision = recall = int(norm_pred == norm_truth)
    elif num_same == 0:
        f1 = precision = recall = 0.0
    else:
        precision = num_same / len(norm_pred)
        recall = num_same / len(norm_truth)
        f1 = (2 * precision * recall) / (precision + recall)

    return exact_match, round(precision, 4), round(recall, 4), round(f1, 4)

def calc_lexical_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0, 0, 0, 0, 0, 0
    smoothie = SmoothingFunction().method4
    ref = [normalize_text(truth).split()]
    hyp = normalize_text(pred).split()
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    b4 = sentence_bleu(ref, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(str(truth), str(pred))

    return round(b1,4), round(b2,4), round(b4,4), \
           round(rouge_scores['rouge1'].fmeasure,4), \
           round(rouge_scores['rouge2'].fmeasure,4), \
           round(rouge_scores['rougeL'].fmeasure,4)

def calc_semantic_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0,0,0,0
    P, R, F1 = bert_score([str(pred)], [str(truth)], lang="en", model_type="distilbert-base-uncased", device="cpu")

    embeddings = metric_model.encode([str(pred), str(truth)], convert_to_tensor=True)
    cosine_sim = float(util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0])

    return round(P.item(),4), round(R.item(),4), round(F1.item(),4), round(cosine_sim,4)

def calc_rag_metrics_llm(query, context, response, expected):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert RAG evaluator. Grade the response based on the following three metrics. "
                "Output ONLY three comma-separated numbers between 0 and 10 (e.g., '9, 8, 10'). Do not output any other text.\n"
                "1. Faithfulness: Is the Response completely grounded in the Context? (10=fully grounded, 0=hallucinated)\n"
                "2. Answer Relevancy: Does the Response address the Query? (10=perfectly answers, 0=irrelevant)\n"
                "3. Context Recall: Is the Expected Answer found within the Context? (10=fully found, 0=missing)"
            )
        },
        {
            "role": "user",
            "content": f"Query: {query}\nExpected Answer: {expected}\nContext: {context}\nResponse: {response}"
        }
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=15, temperature=0.01)

    input_length = inputs["input_ids"].shape[1]
    score_text = tokenizer.decode(output[0][input_length:], skip_special_tokens=True).strip()

    del inputs, output
    torch.cuda.empty_cache()

    try:
        scores = [int(re.search(r'\d+', s).group()) for s in score_text.split(',')]
        if len(scores) >= 3:
            return scores[0], scores[1], scores[2]
    except Exception:
        pass
    return 0, 0, 0

# ── RAG query helper ──────────────────────────────────────────────────────────
def rag_query(query: str, expected: str):
    # Retrieve
    candidates = retriever.invoke(query)

    # Rerank and keep top 4
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[:4]]

    context = "\n\n".join(doc.page_content for doc in top_docs)
    pages = ", ".join(str(doc.metadata.get("page", "?")) for doc in top_docs)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical expert assistant specialising in cardiology. "
                "Answer the user's question using ONLY the context provided below. "
                "If the answer is not in the context, say you don't know.\n\n"
                f"Context:\n{context}"
            ),
        },
        {"role": "user", "content": query},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 10. System Metrics Tracking
    gen_start = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            repetition_penalty=1.05,
        )
    gen_time = time.time() - gen_start

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = len(generated_ids[0]) - input_length
    tokens_per_sec = generated_tokens / gen_time if gen_time > 0 else 0

    answer = tokenizer.decode(generated_ids[0][input_length:], skip_special_tokens=True)

    del inputs, generated_ids
    torch.cuda.empty_cache()

    # Calculate Metrics
    exact_match, tok_p, tok_r, tok_f1 = calc_accuracy_metrics(answer, expected)
    bleu1, bleu2, bleu4, rouge1, rouge2, rougeL = calc_lexical_metrics(answer, expected)
    bert_p, bert_r, bert_f1, cos_sim = calc_semantic_metrics(answer, expected)
    faithfulness, answer_rel, ctx_recall = calc_rag_metrics_llm(query, context, answer, expected)

    return {
        "Response": answer,
        "Pages": pages,
        "Exact Match": exact_match,
        "Token F1": tok_f1, "Token P": tok_p, "Token R": tok_r,
        "BLEU-1": bleu1, "BLEU-2": bleu2, "BLEU-4": bleu4,
        "ROUGE-1": rouge1, "ROUGE-2": rouge2, "ROUGE-L": rougeL,
        "BERTScore F1": bert_f1, "BERTScore P": bert_p, "BERTScore R": bert_r,
        "Semantic Similarity": cos_sim,
        "Faithfulness": faithfulness,
        "Answer Relevancy": answer_rel,
        "Context Recall": ctx_recall,
        "Latency (s)": round(gen_time, 2),
        "Tokens/sec": round(tokens_per_sec, 2)
    }

# ── Run all prompts & write CSV ───────────────────────────────────────────────
try:
    print(f"📂 Loading prompts from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
except FileNotFoundError:
    print(f"❌ Error: Could not find {INPUT_CSV}.")
    df = pd.DataFrame()

if not df.empty:
    print("=" * 70)
    print(f"Running Qwen RAG evaluation over {len(df)} test prompts...")
    print("=" * 70)

    headers = [
        "Query", "Expected", "Response", "Pages",
        "Exact Match", "Token F1", "Token P", "Token R",
        "BLEU-1", "BLEU-2", "BLEU-4", "ROUGE-1", "ROUGE-2", "ROUGE-L",
        "BERTScore F1", "BERTScore P", "BERTScore R", "Semantic Similarity",
        "Faithfulness", "Answer Relevancy", "Context Recall",
        "Latency (s)", "Tokens/sec"
    ]

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()

    for index, row in df.iterrows():
        query = row['Prompt (Question)']
        expected = row['Ground Truth (Expected Answer)']

        print(f"\n[{index+1:02d}/{len(df)}] {query}")

        res = rag_query(query, expected)

        print(f"  RAG Scores (Fth/Rel/Ctx): {res['Faithfulness']},{res['Answer Relevancy']},{res['Context Recall']} | F1: {res['Token F1']} | Speed: {res['Tokens/sec']} T/s")
        print(f"  Answer: {res['Response'][:150]}{'...' if len(res['Response']) > 150 else ''}")

        row_data = {
            "Query": query, "Expected": expected, "Response": res['Response'], "Pages": res['Pages'],
            "Exact Match": res['Exact Match'], "Token F1": res['Token F1'], "Token P": res['Token P'], "Token R": res['Token R'],
            "BLEU-1": res['BLEU-1'], "BLEU-2": res['BLEU-2'], "BLEU-4": res['BLEU-4'],
            "ROUGE-1": res['ROUGE-1'], "ROUGE-2": res['ROUGE-2'], "ROUGE-L": res['ROUGE-L'],
            "BERTScore F1": res['BERTScore F1'], "BERTScore P": res['BERTScore P'], "BERTScore R": res['BERTScore R'],
            "Semantic Similarity": res['Semantic Similarity'],
            "Faithfulness": res['Faithfulness'], "Answer Relevancy": res['Answer Relevancy'], "Context Recall": res['Context Recall'],
            "Latency (s)": res['Latency (s)'], "Tokens/sec": res['Tokens/sec']
        }

        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writerow(row_data)

        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n✅ Evaluation complete. Results saved to '{OUTPUT_CSV}'.")


⚙️  Loading and indexing PDF with MedCPT...
  Loading MedCPT Query Encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  RAG Scores (Fth/Rel/Ctx): 10,10,10 | F1: 0.8077 | Speed: 7.48 T/s
  Answer: According to the guidelines, permanent AF is defined as AF for which no further attempts at restoration of sinus rhythm are planned, after a shared de...

[20/20] For patients undergoing cardiac surgery, when is surgical closure of the LAA recommended?


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  RAG Scores (Fth/Rel/Ctx): 10,10,10 | F1: 0.6383 | Speed: 7.59 T/s
  Answer: Surgical closure of the left atrial appendage (LAA) is recommended as an adjunct to oral anticoagulation in patients with atrial fibrillation (AF) und...

✅ Evaluation complete. Results saved to 'rag_evaluation_qwen_full_metrics.csv'.
